In [2]:
import os
import numpy as np
import tensorflow as tf
import joblib
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [3]:
# Descargar YAMNet si no lo tienes
yamnet_model = tf.keras.models.load_model("https://tfhub.dev/google/yamnet/1", compile=False)


UnimplementedError: File system scheme 'https' not implemented (file: 'https://tfhub.dev/google/yamnet/1')

In [ ]:
def extract_features(audio_path):
    waveform, sr = librosa.load(audio_path, sr=16000)
    waveform = waveform.astype(np.float32)
    _, embeddings, _ = yamnet_model(waveform)
    return np.mean(embeddings.numpy(), axis=0)  # vector de 1024 dims


In [ ]:
data = []
labels = []

audio_paths = ["audio1.wav", "audio2.wav"]   # RUTAS REALES
audio_labels = [0, 1]                        # TUS CLASES

for path, label in zip(audio_paths, audio_labels):
    feats = extract_features(path)
    data.append(feats)
    labels.append(label)

X = np.array(data)
y = np.array(labels)


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

joblib.dump(scaler, "scaler.joblib")   # IMPORTANTE para SageMaker
print("Scaler guardado como scaler.joblib")


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_scaled.shape[1],)),
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(len(np.unique(y)), activation="softmax")
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.fit(X_scaled, y, epochs=30, batch_size=16)


In [ ]:
export_dir = "exported_model/1"
os.makedirs(export_dir, exist_ok=True)

tf.saved_model.save(model, export_dir)
print("Modelo exportado a:", export_dir)


In [ ]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("exported_model", arcname="exported_model")
    tar.add("scaler.joblib", arcname="scaler.joblib")

print("model.tar.gz creado correctamente")
